## 1. Data

In [ ]:
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import re
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from keras_hub.layers import TransformerDecoder, TokenAndPositionEmbedding
import sentencepiece as spm
import kagglehub

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Speedrun for RTX 3080 COMMENT THIS IF NEEDED
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")


2026-05-07 11:51:50.495032: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-07 11:51:50.504937: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-07 11:51:50.504952: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-07 11:51:50.511755: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-07 11:51:50.942663: W tensorflow/co

In [ ]:
# Can probably be commented out if you have no problems with GPU / CUDA!!!
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))
tf.config.optimizer.set_jit(True)               # SET THIS TO FALSE IF NEEDED
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

2.16.2
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2026-05-07 11:51:51.521494: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-07 11:51:51.537756: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-07 11:51:51.539297: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [3]:
path = kagglehub.dataset_download("rohitgr/wikitext")
wikitext2_path = os.path.join(path, "wikitext-103")

def load_lines(base_path, filename):
    with open(os.path.join(base_path, filename), "r", encoding="utf-8") as f:
        return f.readlines()


train_lines = load_lines(wikitext2_path, "wiki.train.tokens")
val_lines   = load_lines(wikitext2_path, "wiki.valid.tokens")
test_lines  = load_lines(wikitext2_path, "wiki.test.tokens")

## 2. Data Cleaning

In [4]:
def clean_line(text_line):
    text_line = text_line.strip()

    # Drop WikiText section titles; they are metadata, not natural sentences.
    if re.match(r"^=+\s*[^=]+?\s*=+$", text_line):
        return ""

    text_line = text_line.replace("<unk>", "")
    text_line = text_line.replace("@-@", "-")
    text_line = text_line.replace("@,@", ",")
    text_line = text_line.replace("@.@", ".")
    text_line = re.sub(r"\s+([,.;:!?])", r"\1", text_line)
    text_line = re.sub(r"([([{])\s+", r"\1", text_line)
    text_line = re.sub(r"\s+([)\]}])", r"\1", text_line)
    text_line = re.sub(r"\s+", " ", text_line)
    return text_line.strip()


train_lines_clean = [clean_line(text_line) for text_line in train_lines]
val_lines_clean   = [clean_line(text_line) for text_line in val_lines]
test_lines_clean  = [clean_line(text_line) for text_line in test_lines]

train_lines_clean = [line for line in train_lines_clean if len(line.split()) >= 4]
val_lines_clean   = [line for line in val_lines_clean if len(line.split()) >= 4]
test_lines_clean  = [line for line in test_lines_clean if len(line.split()) >= 4]


## 3. Subword Tokenizer

In [5]:
with open("wikitext_train.txt", "w", encoding="utf-8") as f:
    for line in train_lines_clean:
        f.write(line + "\n")

In [6]:
vocab_size_limit = 25000

if not os.path.exists("spm_wikitext.model"):
    for f in ["spm_wikitext.model", "spm_wikitext.vocab"]:
        if os.path.exists(f):
            os.remove(f)

    spm.SentencePieceTrainer.train(
        input="wikitext_train.txt",
        model_prefix="spm_wikitext",
        vocab_size=vocab_size_limit,
        model_type="bpe",
        character_coverage=1.0,
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
        input_sentence_size=500000,
    )

sp = spm.SentencePieceProcessor()
sp.load("spm_wikitext.model")


True

In [7]:
eos = sp.eos_id()

train_sequences = sp.encode(train_lines_clean, out_type=int)
val_sequences   = sp.encode(val_lines_clean, out_type=int)
test_sequences  = sp.encode(test_lines_clean, out_type=int)

In [8]:
train_sequences = [seq + [eos] for seq in train_sequences]
val_sequences   = [seq + [eos] for seq in val_sequences]
test_sequences  = [seq + [eos] for seq in test_sequences]

train_sequence = [tok for seq in train_sequences for tok in seq]
val_sequence   = [tok for seq in val_sequences for tok in seq]
test_sequence  = [tok for seq in test_sequences for tok in seq]

vocab_size = sp.get_piece_size()

print("Train tokens:", len(train_sequence))
print("Val tokens:", len(val_sequence))
print("Test tokens:", len(test_sequence))
print("Vocab size:", vocab_size)

Train tokens: 125741485
Val tokens: 260008
Test tokens: 294613
Vocab size: 12000


## 4. Dataset and Hyperparameters

In [9]:
# Hyperparameters
batch_size = 64
train_stride = 64
val_stride = 64
test_stride = 64
seq_len = 64

MAX_TRAIN_BATCHES = 4000  # Set to an integer, e.g. 4000, for a quicker training run

train_tokens = np.array(train_sequence, dtype=np.int32)
val_tokens   = np.array(val_sequence, dtype=np.int32)
test_tokens  = np.array(test_sequence, dtype=np.int32)


def make_dataset(tokens, seq_len, batch_size, shuffle=False, stride=64):
    tokens = np.asarray(tokens, dtype=np.int32)

    ds = tf.keras.utils.timeseries_dataset_from_array(
        data=tokens,
        targets=None,
        sequence_length=seq_len + 1,
        sequence_stride=stride,
        batch_size=batch_size,
        shuffle=False,
    )

    ds = ds.map(lambda window: (window[:, :-1], window[:, 1:]), num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
    return ds.prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(
    train_tokens,
    seq_len,
    batch_size,
    shuffle=True,
    stride=train_stride,
)

if MAX_TRAIN_BATCHES is not None:
    train_ds = train_ds.take(MAX_TRAIN_BATCHES)

val_ds = make_dataset(
    val_tokens,
    seq_len,
    batch_size,
    stride=val_stride,
)

test_ds = make_dataset(
    test_tokens,
    seq_len,
    batch_size,
    stride=test_stride,
)


2026-05-07 11:52:52.351648: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-07 11:52:52.354853: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-07 11:52:52.357214: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [10]:
print("characters:", len(train_lines_clean))
print("tokens:", len(train_tokens))
print("steps:", tf.data.experimental.cardinality(train_ds).numpy())

characters: 1110788
tokens: 125741485
steps: 4000


## 5. Model

In [11]:
embedding_dim = 128
embed_multiplier = 2
num_heads = 4
decoder_amount = 2
dropout_rate = 0.1

inputs = keras.Input(shape=(None,), dtype="int64", name="inputs")
x = TokenAndPositionEmbedding(
    vocabulary_size=vocab_size,
    sequence_length=seq_len,
    embedding_dim=embedding_dim,
    mask_zero=False,
)(inputs)

for _ in range(decoder_amount):
    x = TransformerDecoder(
        intermediate_dim=embedding_dim * embed_multiplier,
        num_heads=num_heads,
        dropout=dropout_rate,
    )(x)

x = layers.Dropout(dropout_rate)(x)
# outputs = layers.Dense(vocab_size)(x)  # SparseCategoricalCrossentropy uses logits directly.
outputs = layers.Dense( # changed to this
    vocab_size,
    dtype="float32"
)(x)

transformer_model = keras.Model(inputs, outputs, name="decoder_only_transformer")


In [12]:
transformer_model.summary()

Model: "decoder_only_transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inputs (InputLayer)             │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, None, 128)      │     1,544,192 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder             │ (None, None, 128)      │       132,480 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_1           │ (None, None, 128)      │       132,480 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, None, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 12000)    │     1,548,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,357,152 (12.81 MB)

 Trainable params: 3,357,152 (12.81 MB)

 Non-trainable params: 0 (0.00 B)

## 6. Training

In [13]:
USE_EARLY_STOPPING = True


def make_callbacks(ckpt_path, use_early_stopping=USE_EARLY_STOPPING):
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            ckpt_path,
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=1,
        ),
    ]

    if use_early_stopping:
        callbacks.append(
            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                mode="min",
                patience=4,
                min_delta=0.001,
                start_from_epoch=4,
                restore_best_weights=True,
                verbose=1,
            )
        )

    return callbacks


In [14]:
class Perplexity(keras.metrics.Metric):
    def __init__(self, name="perplexity", **kwargs):
        super().__init__(name=name, **kwargs)
        self.loss_tracker = keras.metrics.Mean()

    def update_state(self, y_true, y_pred, sample_weight=None):
        loss = keras.losses.sparse_categorical_crossentropy(
            y_true, y_pred, from_logits=True
        )

        self.loss_tracker.update_state(loss, sample_weight=sample_weight)

    def result(self):
        return tf.exp(self.loss_tracker.result())

    def reset_state(self):
        self.loss_tracker.reset_state()

In [15]:
epochs = 24

# The saved model
# if os.path.exists("best_model.keras"):
#     transformer_model = keras.models.load_model(
#     "best_model.keras",
#     compile=False
# )
    
# transformer_model.compile(
#     loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
#     metrics=[
#         "accuracy",
#         keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top_5_accuracy"),
#         Perplexity(),
#     ],
# )

# THE ACTUAL TRAINABLE MODEL DO NOT REMOVE!!

transformer_model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.AdamW(
        learning_rate=2e-4,
        weight_decay=2e-4,
        clipnorm=1.0,
    ),
    metrics=[
        "accuracy",
        keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top_5_accuracy"),
        Perplexity(),
    ],
    jit_compile=False,
)

transformer_history = transformer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=make_callbacks("best_model_new.keras"),
)


Epoch 1/24


I0000 00:00:1778143975.721612   72290 service.cc:145] XLA service 0x7fb2a80050e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778143975.721629   72290 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 3080, Compute Capability 8.6
2026-05-07 11:52:55.736897: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
I0000 00:00:1778143975.750352   72290 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-05-07 11:52:58.992735: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778143981.608392   72584 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_5', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1778143981.679099   72586 asm_compiler.cc

4000/4000 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1114 - loss: 6.7048 - perplexity: 1080.6020 - top_5_accuracy: 0.2331

I0000 00:00:1778144059.080050   73476 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_5', 12 bytes spill stores, 12 bytes spill loads

I0000 00:00:1778144059.196919   73475 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_5', 56 bytes spill stores, 56 bytes spill loads

I0000 00:00:1778144059.216055   73484 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_5', 44 bytes spill stores, 44 bytes spill loads




Epoch 1: val_loss improved from None to 5.36458, saving model to best_model_new.keras

Epoch 1: finished saving model to best_model_new.keras
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 89s 19ms/step - accuracy: 0.1436 - loss: 6.0711 - perplexity: 433.1438 - top_5_accuracy: 0.2879 - val_accuracy: 0.1836 - val_loss: 5.3646 - val_perplexity: 213.7012 - val_top_5_accuracy: 0.3553 - learning_rate: 2.0000e-04
Epoch 2/24
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1781 - loss: 5.3508 - perplexity: 211.0561 - top_5_accuracy: 0.3543

I0000 00:00:1778144130.340833   74412 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_5', 12 bytes spill stores, 12 bytes spill loads

I0000 00:00:1778144130.443055   74395 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_5', 56 bytes spill stores, 56 bytes spill loads

I0000 00:00:1778144130.536398   74396 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_12', 52 bytes spill stores, 52 bytes spill loads

I0000 00:00:1778144130.692923   74403 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_12', 36 bytes spill stores, 40 bytes spill loads

I0000 00:00:1778144130.711449   74398 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_12', 36 bytes spill stores, 36 bytes spill loads

I0000 00:00:1778144130.731038   74405 asm_


Epoch 2: val_loss improved from 5.36458 to 5.03504, saving model to best_model_new.keras

Epoch 2: finished saving model to best_model_new.keras
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 69s 17ms/step - accuracy: 0.1815 - loss: 5.2721 - perplexity: 194.8324 - top_5_accuracy: 0.3608 - val_accuracy: 0.1984 - val_loss: 5.0350 - val_perplexity: 153.7063 - val_top_5_accuracy: 0.3836 - learning_rate: 2.0000e-04
Epoch 3/24
3997/4000 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1911 - loss: 5.0915 - perplexity: 162.6892 - top_5_accuracy: 0.3779
Epoch 3: val_loss improved from 5.03504 to 4.88100, saving model to best_model_new.keras

Epoch 3: finished saving model to best_model_new.keras
4000/4000 ━━━━━━━━━━━━━━━━━━━━ 62s 15ms/step - accuracy: 0.1930 - loss: 5.0594 - perplexity: 157.4887 - top_5_accuracy: 0.3804 - val_accuracy: 0.2112 - val_loss: 4.8810 - val_perplexity: 131.7630 - val_top_5_accuracy: 0.3999 - learning_rate: 2.0000e-04
Epoch 4/24
3998/4000 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy:

In [16]:
for loss in transformer_history.history["val_loss"]:
    print(np.exp(loss))

213.7011885134572
153.70623866343433
131.76293137990316
120.53810915223538
113.02781660835345
108.01298521831393
103.90722365029522
100.07031690295749
97.51187588153745
97.34393298216202
95.43268548756771
93.74120378614886
92.17665629585807
91.02803411258053
90.64907000349861
89.90890039422288
89.87533795883547
88.66393999189934
88.37939978549792
87.23361787557123
86.61652620201983
86.01146901839473
85.53616965815421
85.62145661143572


In [17]:
test_results = transformer_model.evaluate(test_ds, verbose=1)

for name, value in zip(transformer_model.metrics_names, test_results):
    print(f"{name}: {value:.4f}")

72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.2563 - loss: 4.4314 - perplexity: 84.0463 - top_5_accuracy: 0.4561
loss: 4.4314
compile_metrics: 0.2563


## 7. Sampling

In [18]:
def sample_probs(logits, temperature=1.0):
    temperature = max(float(temperature), 1e-6)
    logits = np.asarray(logits).astype("float64") / temperature
    logits = logits - np.max(logits)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits)


def top_p_filter(probs, top_p=0.9, min_tokens_to_keep=3):
    sorted_idx = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_idx]
    cumulative = np.cumsum(sorted_probs)

    keep = cumulative <= top_p
    keep[:min_tokens_to_keep] = True
    filtered_idx = sorted_idx[keep]
    filtered_probs = probs[filtered_idx]
    filtered_probs = filtered_probs / filtered_probs.sum()
    return filtered_idx, filtered_probs


In [19]:
BAD_PIECES = {
    "<pad>", "<unk>", "<s>",
    "<", ">", "▁<", "▁>",
    "__&", "__/",
    "@-@", "▁@-@", " "
}

BAD_IDS = {
    sp.pad_id(),
    sp.unk_id(),
    sp.bos_id(),
}

for piece in BAD_PIECES:
    piece_id = sp.piece_to_id(piece)
    if piece_id != sp.unk_id():
        BAD_IDS.add(piece_id)


def encode_prompt(text):
    seq = sp.encode(text, out_type=int)
    seq = seq[-seq_len:]
    return np.array([seq], dtype=np.int32)


def repetition_adjusted_logits(logits, generated_ids, penalty=1.2, recent_window=80, hard_window=6):
    logits = np.array(logits, dtype="float64", copy=True)

    for bad_id in BAD_IDS:
        if 0 <= bad_id < len(logits):
            logits[bad_id] = -1e9

    recent_ids = generated_ids[-recent_window:]
    hard_block_ids = set(generated_ids[-hard_window:])

    for token_id in set(recent_ids):
        if 0 <= token_id < len(logits):
            if token_id in hard_block_ids:
                logits[token_id] = -1e9
            elif logits[token_id] > 0:
                logits[token_id] /= penalty
            else:
                logits[token_id] *= penalty

    return logits


def predict_top_n(text, model, top_n=5, temperature=0.8, filter_bad=True):
    seq = encode_prompt(text)
    logits = model.predict(seq, verbose=0)[0, -1]

    if filter_bad:
        logits = repetition_adjusted_logits(logits, sp.encode(text, out_type=int), penalty=1.05)

    probs = sample_probs(logits, temperature)
    sorted_indices = np.argsort(probs)[::-1]

    results = []
    for token_id in sorted_indices:
        piece = sp.id_to_piece(int(token_id))
        decoded = sp.decode([int(token_id)])
        if not decoded.strip() and piece != "▁":
            continue

        results.append((decoded, round(float(probs[token_id]) * 100, 2)))
        if len(results) == top_n:
            break

    return results


def sample_next_token_id(generated_ids, model, temperature=0.8, top_k=50, top_p=0.9):
    seq = np.array([generated_ids[-seq_len:]], dtype=np.int32)
    logits = model.predict(seq, verbose=0)[0, -1]
    logits = repetition_adjusted_logits(logits, generated_ids)

    probs = sample_probs(logits, temperature)
    top_indices = np.argsort(probs)[::-1][:top_k]
    top_probs = probs[top_indices]
    top_probs = top_probs / top_probs.sum()

    if top_p is not None:
        kept_local_idx, kept_probs = top_p_filter(top_probs, top_p=top_p)
        top_indices = top_indices[kept_local_idx]
        top_probs = kept_probs

    return int(np.random.choice(top_indices, p=top_probs))


def generate_text(seed_text, model, num_tokens=60, temperature=0.75, top_k=60, top_p=0.9):
    generated_ids = sp.encode(seed_text, out_type=int)

    for _ in range(num_tokens):
        next_id = sample_next_token_id(
            generated_ids,
            model,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
        )

        if next_id == sp.eos_id():
            break

        generated_ids.append(next_id)

    text = sp.decode(generated_ids)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [21]:
# Tuneable settings for making predictions
number_of_tokens = 40
settings = [
    ("safe", 0.35, 15, 0.8),
    ("balanced", 0.6, 40, 0.9),
    ("creative", 0.8, 80, 0.95),
]

seeds = seeds = [
    "The film received generally",
    "The album was released in",
    "The war ended in",
    "The character is described as",
    "in her early years, she was",
    "the city is located in",
    "the population of the town was",
    "according to the report",
]

print("\n=== Transformer: Top-5 seuraavaa sanaa ===")
for i in range(len(seeds)):
    print("------------------")
    for piece, pct in predict_top_n(seeds[i], transformer_model, top_n=5):
        print(f"  {piece:20s} {pct:.2f}%")

print("\n=== Transformer generoitu teksti ===")
for name, temp, top_k, top_p in settings:
    print("------------------")
    for seed in seeds:
        print()
        print(generate_text(
            seed,
            transformer_model,
            num_tokens=number_of_tokens,
            temperature=temp,
            top_k=top_k,
            top_p=top_p,
        ))



=== Transformer: Top-5 seuraavaa sanaa ===
------------------
  positive             82.27%
  favorable            12.05%
  mixed                2.87%
  negative             2.36%
  reviews              0.08%
------------------
  the                  25.58%
  November             5.17%
  October              3.85%
  Japan                3.75%
  September            3.73%
------------------
  the                  22.04%
  a                    4.60%
  September            4.32%
  February             3.24%
  December             3.18%
------------------
  "                    45.43%
  a                    33.98%
  the                  7.51%
  an                   4.22%
  being                2.09%
------------------
  born                 7.74%
  a                    5.74%
  not                  4.16%
  able                 3.40%
  the                  2.13%
------------------
  a                    13.42%
  downtown             4.89%
  New                  3.25%
  an                   